# Dataset: Tensor "Highway" On-Ramp

- TODO: expand bullets

- complex input data pipelines from simple, reusable pieces
- while computers were made to add numbers together, they quickly run into insurmountable obstacles if these numbers are presented as textual sequences of digits
- we aim to finally "teach" our computer to correctly add and multiply
- use example - fascinating https://arxiv.org/pdf/1812.02825.pdf

In [1]:
import numpy as np
import pathlib as pth
import tensorflow as tf

- more preps

In [2]:
td = tf.data
tt = tf.train

- our data consists of `num_samples` (perhaps easily millions) of `"x=-12,y=24:y+x:12"`-like strings of texts with `defs`, `op` and `res` fields (separated by `:`)
- our variables are: `x` and `y`
- our "operations" are: `+`, `-` and `*`
- and our variables can be assigned values from: `[-max_val, max_val]`

In [3]:
vocab = ('x', 'y')
vocab += ('+', '-', '*', '=', ',', ':')
vocab += ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

- our input data pipeline is parametric, without error-prone string names

In [4]:
params = dict(
    max_val=10,
    num_samples=4,
    num_shards=3,
)

class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

- let's generate our data randomly based on our given `Params` instance as a Python generator

In [5]:
def py_gen(ps):
    m, n = ps.max_val, ps.num_samples
    # x, y vals in defs
    vals = np.random.randint(low=-m, high=m + 1, size=(2, n))
    # (x, y) order if 1 in defs [0] and op [1], respectively
    ords = np.random.randint(2, size=(2, n))
    # index of ['+', '-', '*']
    ops = np.array(['+', '-', '*'])
    ops.reshape((1, 3))
    ops = ops[np.random.randint(3, size=n)]
    for i in range(n):
        x, y = vals[:, i]
        res = f'x={x},y={y}:' if ords[0, i] else f'y={y},x={x}:'
        o = ops[i]
        res += (f'x{o}y:' if ords[1, i] else f'y{o}x:')
        if o == '+':
            res += f'{x + y}'
        elif o == '*':
            res += f'{x * y}'
        else:
            assert o == '-'
            res += (f'{x - y}' if ords[1, i] else f'{y - x}')
        yield res

- ok, let's do it!

In [9]:
ps = Params(**params)
for s in py_gen(ps):
    print(s)

y=9,x=2:x*y:18
x=-7,y=9:x*y:-63
y=1,x=7:x*y:7
x=-5,y=8:y+x:3


- the class `tf.data.Dataset`is an abstraction for a sequence of elements
- each element is one or more Tensors containing our data
- an example is the following dataset directly using our "in-memory" generator

In [6]:
def gen_src(ps):
    ds = td.Dataset.from_generator(
        lambda: py_gen(ps),
        tf.string,
        tf.TensorShape([]),
    )
    return ds

- and here is the first 2 samples of the now tensor-based sequence

In [10]:
dg = gen_src(ps)
for s in dg.take(2):
    print(s)

tf.Tensor(b'x=8,y=-7:x-y:15', shape=(), dtype=string)
tf.Tensor(b'x=-3,y=-1:x-y:-2', shape=(), dtype=string)


- an input data pipeline starts with a "source" `dataset`
- this "source" can also be a `range`, `from_tensor_slices`, `from_tensors` and even `TextLineDataset`

In [7]:
def src_dset(ps):
    ds = np.array(list(py_gen(ps)))
    ds = td.Dataset.from_tensor_slices(ds)
    return ds

- all `datasets` can be consumed as iterables or as aggregatables (e.g. reduce)
- they also allow chaining of "transformations" to themselves
- some of the handy canned operations: *cache, concatenate, enumerate, reduce, repeat, shuffle, skip, take, zip*
- an example of 2 samples of a new dataset, concatenated with all 4 samples of the previous, gen-based, dataset and "enumerated" is as follows:

In [11]:
ds = src_dset(ps)
for i, s in ds.take(2).concatenate(dg).enumerate():
    print(i, s)

tf.Tensor(0, shape=(), dtype=int64) tf.Tensor(b'x=-5,y=-9:x-y:4', shape=(), dtype=string)
tf.Tensor(1, shape=(), dtype=int64) tf.Tensor(b'x=6,y=9:y-x:3', shape=(), dtype=string)
tf.Tensor(2, shape=(), dtype=int64) tf.Tensor(b'x=-9,y=5:x*y:-45', shape=(), dtype=string)
tf.Tensor(3, shape=(), dtype=int64) tf.Tensor(b'y=5,x=-9:y+x:-4', shape=(), dtype=string)
tf.Tensor(4, shape=(), dtype=int64) tf.Tensor(b'y=-6,x=-5:x-y:1', shape=(), dtype=string)
tf.Tensor(5, shape=(), dtype=int64) tf.Tensor(b'y=-9,x=-10:y*x:90', shape=(), dtype=string)


- we can also filter the "pipeline" at any stage:

In [12]:
@tf.function
def filterer(x):
    r = tf.strings.length(x) < 15
    tf.print(tf.strings.format('filtering {}... ', x) + ('in' if r else 'out'))
    return r

for i, s in enumerate(ds.filter(filterer)):
    print(i, s)

filtering "x=-5,y=-9:x-y:4"... out
filtering "x=6,y=9:y-x:3"... in
0 tf.Tensor(b'x=6,y=9:y-x:3', shape=(), dtype=string)
filtering "y=-10,x=-10:x*y:100"... out
filtering "y=-1,x=8:y-x:-9"... out


- even more importantly, we can split the pipeline into named "filaments" of data
- this new feature proves to be extremely useful, allowing us to standardize and unify our data sources with configurable-on-the-fly channeling of the features aggregated therein

In [14]:
@tf.function
def splitter(x):
    fs = tf.strings.split(x, ':')
    return {'defs': fs[0], 'op': fs[1], 'res': fs[2]}

for s in ds.map(splitter).take(1):
    print(s)

{'defs': <tf.Tensor: id=205, shape=(), dtype=string, numpy=b'x=-5,y=-9'>, 'op': <tf.Tensor: id=206, shape=(), dtype=string, numpy=b'x-y'>, 'res': <tf.Tensor: id=207, shape=(), dtype=string, numpy=b'4'>}


- another example of a "pipeline component" is an in-line Python `dict`-based tokenizer:

In [16]:
tokens = {k: v for v, k in enumerate(vocab, start=5)}

@tf.function
def tokenizer(d):
    return {
        k: tf.numpy_function(
            lambda x: tf.constant([tokens[chr(c)] for c in x]),
            [v],
            Tout=tf.int32,
        )
        for k, v in d.items()
    }

for s in ds.map(splitter).map(tokenizer).take(1):
    print(s)

{'defs': <tf.Tensor: id=256, shape=(9,), dtype=int32, numpy=array([ 5, 10,  8, 18, 11,  6, 10,  8, 22], dtype=int32)>, 'op': <tf.Tensor: id=257, shape=(3,), dtype=int32, numpy=array([5, 8, 6], dtype=int32)>, 'res': <tf.Tensor: id=258, shape=(1,), dtype=int32, numpy=array([17], dtype=int32)>}


- `dataset`s can be potentially very large, fitting only on disk in many files
- as transparent performance is key for training throughput, datasets can be efficiently encoded into binary sequences stored in sharded files
- the following will convert our samples into binary "records"

In [17]:
def records(dset):
    for s in dset:
        fs = tt.Features(
            feature={
                'defs': tt.Feature(int64_list=tt.Int64List(value=s['defs'])),
                'op': tt.Feature(int64_list=tt.Int64List(value=s['op'])),
                'res': tt.Feature(int64_list=tt.Int64List(value=s['res'])),
            })
        yield tt.Example(features=fs).SerializeToString()

- and we can "dump" our tokenized, ready-to-consume samples into shards of files stored in a directory
- once these prepared samples are stored we can "stream" them without any more prep into subsequent blogs

In [18]:
def shards(ps):
    for _ in range(ps.num_shards):
        yield src_dset(ps).map(splitter).map(tokenizer)

def dump(ps):
    d = pth.Path('/tmp/q/dataset')
    d.mkdir(parents=True, exist_ok=True)
    for i, ds in enumerate(shards(ps)):
        i = '{:0>4d}'.format(i)
        p = str(d / f'shard_{i}.tfrecords')
        print(f'dumping {p}...')
        with tf.io.TFRecordWriter(p) as w:
            for r in records(ds):
                w.write(r)
        yield p
        
fs = [f for f in dump(ps)]

dumping /tmp/q/dataset/shard_0000.tfrecords...
dumping /tmp/q/dataset/shard_0001.tfrecords...
dumping /tmp/q/dataset/shard_0002.tfrecords...


- for loading the "records" back, we need to create templates for interpreting the stored data
- then loading our samples back straight into a 'dataset' can be just as follows:

In [19]:
features = {
    'defs': tf.io.VarLenFeature(tf.int64),
    'op': tf.io.VarLenFeature(tf.int64),
    'res': tf.io.VarLenFeature(tf.int64),
}

def load(ps, paths):
    ds = td.TFRecordDataset(paths)
    if ps.dim_batch:
        ds = ds.batch(ps.dim_batch)
        return ds.map(lambda x: tf.io.parse_example(x, features))
    return ds.map(lambda x: tf.io.parse_single_example(x, features))

- before we actually load the shards back, let's "adapt" the pipeline to supply dense tensors instead of the originally configured sparse ones (since we haven't batched anything yet, we set `dim_batch` to `None`):

In [21]:
@tf.function
def adapter(d):
    return [
        tf.sparse.to_dense(d['defs']),
        tf.sparse.to_dense(d['op']),
        tf.sparse.to_dense(d['res']),
    ]

ps.dim_batch = None
for i, s in enumerate(load(ps, fs).map(adapter)):
    print(i, len(s))

0 3
1 3
2 3
3 3
4 3
5 3
6 3
7 3
8 3
9 3
10 3
11 3


- the listing above reveals that we merged 3 sharded files, worth 4 samples each, into the 12 samples
- only the number of "features" is printed for each sample for brevity
- **Please also note** how the above in-line adapter converted our named features into unnamed, positional-in-a-list features. This was necessary as the Keras `Input` doesn't recognize named input tensors yet.
- if we turn on batching in our dataset, the same code will now return the following:

In [22]:
ps.dim_batch = 2
for i, s in enumerate(load(ps, fs).map(adapter)):
    print(i, s)

0 (<tf.Tensor: id=1561, shape=(2, 8), dtype=int64, numpy=
array([[ 5, 10, 17, 11,  6, 10, 14,  0],
       [ 5, 10,  8, 18, 11,  6, 10, 22]])>, <tf.Tensor: id=1562, shape=(2, 3), dtype=int64, numpy=
array([[5, 9, 6],
       [5, 8, 6]])>, <tf.Tensor: id=1563, shape=(2, 3), dtype=int64, numpy=
array([[17,  0,  0],
       [ 8, 14, 17]])>)
1 (<tf.Tensor: id=1567, shape=(2, 9), dtype=int64, numpy=
array([[ 5, 10,  8, 20, 11,  6, 10,  8, 18],
       [ 5, 10, 15, 11,  6, 10, 14,  0,  0]])>, <tf.Tensor: id=1568, shape=(2, 3), dtype=int64, numpy=
array([[5, 9, 6],
       [5, 7, 6]])>, <tf.Tensor: id=1569, shape=(2, 2), dtype=int64, numpy=
array([[16, 18],
       [16,  0]])>)
2 (<tf.Tensor: id=1573, shape=(2, 8), dtype=int64, numpy=
array([[ 5, 10, 22, 11,  6, 10, 17,  0],
       [ 6, 10, 19, 11,  5, 10,  8, 20]])>, <tf.Tensor: id=1574, shape=(2, 3), dtype=int64, numpy=
array([[6, 9, 5],
       [5, 7, 6]])>, <tf.Tensor: id=1575, shape=(2, 2), dtype=int64, numpy=
array([[16, 19],
       [ 8, 14]])

- as a preparation for the subsequent blogs, let's generate a more substantial data source with 10 shards of 1,000 samples each:

In [23]:
ps.max_val = 100
ps.num_samples = 1000
ps.num_shards = 10
fs = [f for f in dump(ps)]
ps.dim_batch = 100
for i, _ in enumerate(load(ps, fs).map(adapter)):
    pass
print(i)

dumping /tmp/q/dataset/shard_0000.tfrecords...
dumping /tmp/q/dataset/shard_0001.tfrecords...
dumping /tmp/q/dataset/shard_0002.tfrecords...
dumping /tmp/q/dataset/shard_0003.tfrecords...
dumping /tmp/q/dataset/shard_0004.tfrecords...
dumping /tmp/q/dataset/shard_0005.tfrecords...
dumping /tmp/q/dataset/shard_0006.tfrecords...
dumping /tmp/q/dataset/shard_0007.tfrecords...
dumping /tmp/q/dataset/shard_0008.tfrecords...
dumping /tmp/q/dataset/shard_0009.tfrecords...
99
